<a href="https://colab.research.google.com/github/Abdouramane-qr/mini_projet_data_analysis/blob/Data-Visualization/Data_Visualization_(visualisation_des_donn%C3%A9es).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#🐍 Code Python (Pipeline avec Pandas)

In [4]:
import pandas as pd
import numpy as np

# ================================
# 1. Jeu de données brut (ventes)
# ================================

ventes = pd.DataFrame({
   "date": ["2025-08-16", "2025-08-16", "2025-08-17", "2025-08-18", "2025-08-18"],
    "client_id": [1, 1, 2, 3, 4],
    "produit": ["P001", "P001", "P002", "P003", "P004"],
   "quantite": [1, 1, np.nan, "deux", 1],
    "prix_unitaire": [2500, 2500, 3000, 4000, -5000],
    "montant": [2500, 2500, np.nan, 8000, -5000]


})

# ================================
# 1. Jeu de données brut (clients)
# ================================

clients = pd.DataFrame(
    {
       "client_id": [1, 2, 3, 4],
    "nom": ["Awa", "Oumar", "Fatou", "Mamadou"],
    "pays": ["Burkina", "ML", "Mali", "Burkina Faso"]

    }
)

print("=== Données brutes ===")
print(ventes)





=== Données brutes ===
         date  client_id produit quantite  prix_unitaire  montant
0  2025-08-16          1    P001        1           2500   2500.0
1  2025-08-16          1    P001        1           2500   2500.0
2  2025-08-17          2    P002      NaN           3000      NaN
3  2025-08-18          3    P003     deux           4000   8000.0
4  2025-08-18          4    P004        1          -5000  -5000.0
         date  client_id produit quantite  prix_unitaire  montant
0  2025-08-16          1    P001        1           2500   2500.0
2  2025-08-17          2    P002      NaN           3000      NaN
3  2025-08-18          3    P003     deux           4000   8000.0
4  2025-08-18          4    P004        1          -5000  -5000.0


In [6]:
# ================================
# 2. Nettoyage
# ================================

# a) Supprimer doublons
ventes = ventes.drop_duplicates()

# b) Corriger valeurs manquantes (remplacer NaN par 1 par défaut pour la quantité)

ventes["quantite"]= ventes["quantite"].replace("deux",2) # corriger le texte "deux"
ventes["quantite"]= ventes["quantite"].fillna(1).astype(int) # remplacer les NaN par 1 et convertir en int

# c) Corriger prix négatifs
ventes= ventes[ventes["prix_unitaire"]> 0]

# d) Recalculer montant si manquant

ventes["montant"]= ventes["quantite"] * ventes["prix_unitaire"]

print("\n=== Après Nettoyage ===")
print(ventes)



=== Après Nettoyage ===
         date  client_id produit  quantite  prix_unitaire  montant
0  2025-08-16          1    P001         1           2500     2500
2  2025-08-17          2    P002         1           3000     3000
3  2025-08-18          3    P003         2           4000     8000


/tmp/ipython-input-2480183776.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ventes["quantite"]= ventes["quantite"].replace("deux",2) # corriger le texte "deux"


In [9]:
# ================================
# 3. Transformation
# ================================

# Convertir date en datetime

ventes["date"] = pd.to_datetime(ventes["date"])

ventes["mois"] = ventes["date"].dt.month_name()

ventes["jour_semaine"]= ventes["date"].dt.day_name()

print("\n=== Après Transformation ===")
print(ventes)


=== Après Transformation ===
        date  client_id produit  quantite  prix_unitaire  montant    mois  \
0 2025-08-16          1    P001         1           2500     2500  August   
2 2025-08-17          2    P002         1           3000     3000  August   
3 2025-08-18          3    P003         2           4000     8000  August   

  jour_semaine  
0     Saturday  
2       Sunday  
3       Monday  


In [10]:
# ================================
# 4. Intégration (jointure avec clients)
# ================================

data = ventes.merge(clients, on="client_id", how="left")

# Normalisation des pays

data["pays"] = data["pays"].replace({"Burkina": "Burkina Faso","ML": "Mali" })


print("\n=== Après Intégration ===")
print(data[["date","nom","pays","produit","quantite","montant"]])




=== Après Intégration ===
        date    nom          pays produit  quantite  montant
0 2025-08-16    Awa  Burkina Faso    P001         1     2500
1 2025-08-17  Oumar          Mali    P002         1     3000
2 2025-08-18  Fatou          Mali    P003         2     8000


In [11]:
# ================================
# 5. Réduction (agrégation par pays)
# ================================

resultat = data.groupby("pays")["montant"].sum().reset_index()

print("\n=== Résultat Réduit (ventes par pays) ===")
print(resultat)


=== Résultat Réduit (ventes par pays) ===
           pays  montant
0  Burkina Faso     2500
1          Mali    11000
